In [1]:
import pandas as pd
import numpy as np

TARGETS = ["x"]

In [2]:
results_1l = pd.read_excel("resultados-1l-v2.xlsx")
# results_2l = pd.read_excel("resultados-2l.xlsx")
# results_3l = pd.read_excel("resultados-3l.xlsx")

# results = pd.concat(
#     [results_1l, results_2l,],
#     ignore_index=True )
results = results_1l


In [3]:
results

,model,Neurons,Ld,Lp,reg,seed,R2_ZZx1_x,MSE_ZZx1_x,R2_ZZx2_x,MSE_ZZx2_x,R2_ZZxReto_x,MSE_ZZxReto_x
0,model_arch1_r0.01_Ld0.5_Lp0.5_seed9180,[1],0.5,0.5,0.01,9180,-0.141991,0.695389,-1.396475,0.642502,0.281491,0.682544
1,model_arch1_r0.01_Ld0.5_Lp0.5_seed9416,[1],0.5,0.5,0.01,9416,-0.060728,0.737109,-6.547791,0.665369,0.166992,0.702693
2,model_arch1_r0.01_Ld0.5_Lp0.5_seed6897,[1],0.5,0.5,0.01,6897,-0.115446,0.702389,-1.362201,0.645338,0.304314,0.689414
3,model_arch1_r0.01_Ld0.5_Lp0.5_seed5014,[1],0.5,0.5,0.01,5014,0.211682,0.790802,-4.146584,0.703689,0.490512,0.778408
4,model_arch1_r0.01_Ld0.5_Lp0.5_seed8826,[1],0.5,0.5,0.01,8826,0.070495,0.771482,-5.548225,0.690608,0.276017,0.739335
...,...,...,...,...,...,...,...,...,...,...,...,...
508,model_arch17_r0.9_Ld0.7_Lp0.3_seed5014,[17],0.7,0.3,0.90,5014,0.965689,0.969597,0.522531,0.784020,0.856468,0.955705
509,model_arch17_r0.9_Ld0.7_Lp0.3_seed8826,[17],0.7,0.3,0.90,8826,0.799541,0.955457,0.652135,0.796347,0.967797,0.952656
510,model_arch18_r0.01_Ld0.5_Lp0.5_seed9180,[18],0.5,0.5,0.01,9180,0.984568,0.969191,0.621722,0.786527,0.870113,0.954461
511,model_arch18_r0.01_Ld0.5_Lp0.5_seed9416,[18],0.5,0.5,0.01,9416,0.958444,0.964485,0.526875,0.701803,0.936344,0.956401


In [4]:
# 🔹 categorização dos sets (baseada nos comentários originais)
SETS_CATEGORY = {
    "ZZx1":     "Train",
    "ZZx2":     "Val",
    "ZZxReto":  "Test",
    "ZZy1":     "Test",
    "ZZy2":     "Test",
    "LSG-1":    "Test",
    "LSG-2":    "Test",
    "ZZx1-inv": "Test",
    "ZZx2-inv": "Test",
    "semiCirc": "Test",
}

def col_name(s, target):
    # sanitiza "-" pra "_" pra bater com o nome real da coluna, se for o caso
    return f"R2_{s.replace('-', '_')}_{target}"

best_models_tables = {}
N = 5  # top modelos

w_val = 0.33
w_train = 0.33
w_test = 0.33


for target in TARGETS:

    # 🔹 sets de Train, Val e Test
    train_sets = [s for s, cat in SETS_CATEGORY.items() if cat == "Train"]
    val_sets   = [s for s, cat in SETS_CATEGORY.items() if cat == "Val"]
    test_sets  = [s for s, cat in SETS_CATEGORY.items() if cat == "Test"]

    train_cols = [col_name(s, target) for s in train_sets]
    val_cols   = [col_name(s, target) for s in val_sets]
    test_cols  = [col_name(s, target) for s in test_sets]

    # 🔹 garantir que só usamos colunas existentes
    train_cols = [c for c in train_cols if c in results.columns]
    val_cols   = [c for c in val_cols if c in results.columns]
    test_cols  = [c for c in test_cols if c in results.columns]

    r2_all_cols = train_cols + val_cols + test_cols

    if not r2_all_cols:
        print(f"⚠️ Nenhuma coluna Train/Val/Test encontrada para target={target}, pulando.")
        continue

    df = results.copy()

    # 🔹 remover linhas onde QUALQUER R2 (Train/Val/Test) < 0
    # df = df[(df[r2_all_cols] >= 0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    df["R2_train_mean"] = df[train_cols].mean(axis=1) if train_cols else np.nan
    df["R2_val_mean"]   = df[val_cols].mean(axis=1) if val_cols else np.nan
    df["R2_test_mean"]  = df[test_cols].mean(axis=1) if test_cols else np.nan

    # =========================
    # 🔹 SCORE
    # =========================
    df["R2_std"] = df[r2_all_cols].std(axis=1)

    df["Score"] = (
        w_train * df["R2_train_mean"] +
        w_val   * df["R2_val_mean"] +
        w_test  * df["R2_test_mean"] - 
        0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    df_sorted = df.sort_values(by="Score", ascending=False)
    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    final_cols = ["model", "Neurons"] + r2_all_cols + [
        "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"
    ]
    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 5 MODELOS - x


,model,Neurons,R2_train_mean,R2_val_mean,R2_test_mean,Score
90,model_arch4_r0.01_Ld0.5_Lp0.5_seed9180,[4],0.980198,0.936363,0.940206,0.940306
344,model_arch12_r0.01_Ld0.3_Lp0.7_seed8826,[12],0.942093,0.913877,0.967821,0.929153
190,model_arch7_r0.01_Ld0.3_Lp0.7_seed9180,[7],0.986867,0.899863,0.930667,0.925329
394,model_arch14_r0.01_Ld0.5_Lp0.5_seed8826,[14],0.981587,0.872204,0.957891,0.922101
244,model_arch9_r0.01_Ld0.5_Lp0.5_seed8826,[9],0.964682,0.859588,0.975874,0.917632



📊 MÉTRICAS COMPLETAS - TOP 5 (x)


,model,Neurons,R2_ZZx1_x,R2_ZZx2_x,R2_ZZxReto_x,R2_train_mean,R2_val_mean,R2_test_mean,Score
90,model_arch4_r0.01_Ld0.5_Lp0.5_seed9180,[4],0.980198,0.936363,0.940206,0.980198,0.936363,0.940206,0.940306
344,model_arch12_r0.01_Ld0.3_Lp0.7_seed8826,[12],0.942093,0.913877,0.967821,0.942093,0.913877,0.967821,0.929153
190,model_arch7_r0.01_Ld0.3_Lp0.7_seed9180,[7],0.986867,0.899863,0.930667,0.986867,0.899863,0.930667,0.925329
394,model_arch14_r0.01_Ld0.5_Lp0.5_seed8826,[14],0.981587,0.872204,0.957891,0.981587,0.872204,0.957891,0.922101
244,model_arch9_r0.01_Ld0.5_Lp0.5_seed8826,[9],0.964682,0.859588,0.975874,0.964682,0.859588,0.975874,0.917632


In [5]:
final_table.to_excel("BestModels-otm.xlsx")